# RUN__pdf_ocr_summary — trạng thái parse báo cáo tài chính, theo cổ phiếu

Đọc **toàn bộ** `raw_data/cafef/financials/statements/**/*.csv` (mọi template: `bank`, `corp`, …),
đối chiếu với **chỉ mục PDF** `raw_data/cafef/pdfs/index/` qua chính
`FinancialsBuilder.documents()`, rồi tóm tắt thành một bảng, mỗi dòng một cổ phiếu, **6 cột**:

| cột | nghĩa |
|---|---|
| `exchange` | sàn niêm yết, đọc từ chính cột `exchange` của CSV |
| `complete` | **đã thực sự hoàn thành chưa** — `True` khi kể từ `first_report` trở đi, **mọi quý CÓ FILING** đều đã đọc được ở cả ba statement |
| `first_report` | **quý mở đầu chuỗi filing LIỀN MẠCH**, đi ngược từ filing mới nhất của chính cổ phiếu đó — đọc từ chỉ mục PDF, *không* phụ thuộc vào việc parse được hay không |
| `balance_sheet` | **quý muộn nhất còn thiếu** ở bảng cân đối kế toán — quý *có filing* mà chưa có dòng `pdf` |
| `income_statement` | như trên, cho báo cáo kết quả kinh doanh |
| `cash_flow` | như trên, cho báo cáo lưu chuyển tiền tệ |

Quý hiển thị dạng **`2008-Q4`** — dạng sắp xếp được, và đúng dạng `--quarters` / `QUARTERS` của
`pdf_ocr_job` nhận, nên một ô ở bảng này dán thẳng vào lệnh parse được. (Trên đĩa CSV vẫn ghi
`Q4-2008`; cột `period` gốc được giữ nguyên trong `records`.)

⚠️ **LỖI ĐÃ SỬA 2026-08-30 — MỐC BẮT ĐẦU TỪNG ĐƯỢC LẤY TỪ CHÍNH KẾT QUẢ PARSE.** Bản trước đặt mốc
của mỗi statement là *quý đầu tiên statement đó parse thành công*, nên **mọi lần parse hỏng nằm ở
đầu chuỗi đều tự bị đẩy ra khỏi mẫu số và không bao giờ đếm được**. BID là ca cụ thể: KQKD Q3-2011
`missing`, trong khi bảng cân đối và lưu chuyển tiền tệ của **đúng filing đó**
(`Q3-2011_bao_cao_tai_chinh_hop_nhat_quy_3_nam_2011.pdf`) đọc được ở `onnx@200` — mốc KQKD bị đẩy
lên 2012-Q1, ô hỏng biến mất khỏi mẫu số, và BID hiện `complete = True`. Cùng hình dạng với `SAN-1`:
một thước đo học mẫu số từ chính thứ nó đang đo.

⚠️ **SỬA 2026-08-30 — MỐC PHẢI LIỀN MẠCH, KHÔNG PHẢI "SỚM NHẤT TỪNG CÓ".** Bản trước lấy quý
*sớm nhất có filing quý*, nên một filing lẻ loi ở rất xa quá khứ cũng mở đầu chuỗi, và mọi quý
trống sau nó bị kéo vào mẫu số. Mốc bây giờ đi **ngược từ filing mới nhất**: chạm vào một quý
không có filing nào là đứt, và những gì nằm trước chỗ đứt không thuộc chuỗi nữa. Đo trên 7 cổ
phiếu đã parse, đúng **hai** cái đổi và cả hai đều là hình dạng đó:

| cổ phiếu | chỉ mục PDF | cũ | **mới** | ô rời khỏi mẫu số |
|---|---|---|---|---|
| **ACB** | 2008-Q1, rồi **trống 4 quý**, rồi 2009-Q2 → 2026-Q1 liền mạch | 2008-Q1 | **2009-Q2** | 3 (cả ba statement của 2008-Q1) |
| **BSR** | 2016-Q4, 2017-Q3, 2017-Q4, **thiếu 2018-Q1**, rồi 2018-Q2 → 2020-Q4 | 2017-Q3 | **2018-Q2** | 3 (2017-Q3 BS; 2017-Q4 BS + KQKD) |
| BID · TCB · VIC · VCB · CTG | không có chỗ đứt nào sau mốc | — | **giữ nguyên** | 0 |

⚠️ **SỬA 2026-08-30 — "CÓ NỘP BÁO CÁO QUÝ" ĐỌC TỪ CHỈ MỤC, KHÔNG TỪ FILING ĐƯỢC CHỌN.**
`documents()` trả về đúng một filing mỗi quý và ưu tiên bản báo cáo **năm** đã kiểm toán ở Q4, nên
cờ `annual` của nó nói *"filing được chọn là bản quý"*, không nói *"doanh nghiệp có nộp báo cáo
quý"*. **ACB 2009-Q4 nộp cả hai** và cột cũ vẫn trả lời `False`. Sửa xong: **98 quý đổi cờ trên 7
cổ phiếu, và KHÔNG mốc nào đổi chỗ** — lỗi tiềm ẩn, chỉ cắn khi chuỗi của một cổ phiếu bắt đầu
đúng ở một Q4 như thế, và khi đó nó lặng lẽ lấy ba ô ra khỏi mẫu số.

⚠️ **VÀ BẢNG NÀY ĐẾM NHỮNG QUÝ MÀ `pdf_ocr_job` Ở MẶC ĐỊNH KHÔNG MỞ ĐƯỢC.** Mẫu số ở đây là
`allow_parent=True`; `pdf_ocr_job.plan()` mặc định `False`. Quý nào bản duy nhất tồn tại là báo cáo
**riêng lẻ** thì bị báo thiếu ở đây và bị `plan()` trả lời *"files no document for quarter(s)
[...]"* ở notebook parse — hai câu trả lời đối nhau về một quý mà file PDF đang nằm trên đĩa. Đo
2026-08-30: **ACB 4 quý (2008-Q1, 2009-Q2, 2009-Q3, 2009-Q4 — đúng cả bốn quý ACB còn thiếu)**,
TCB 2, VIC 1, BID/CTG/VCB/BSR 0. Cột `parent_only` ở cell 3 đo nó, danh sách in dưới bảng nói rõ
phải bật `ALLOW_PARENT=True`.

⚠️ **Liền mạch đo trên MỌI filing, còn mốc vẫn là filing QUÝ — hai luật, chạy theo thứ tự.**
`documents()` gộp báo cáo năm đã kiểm toán vào Q4, nên nếu đòi liền mạch trên *riêng* filing quý
thì **mọi năm đều đứt ở Q4** và chuỗi dài nhất còn 3 quý. Ngược lại, nếu lấy luôn quý đầu của
chuỗi làm mốc thì VCB và CTG lùi về 2008-Q4 — một năm chỉ có báo cáo năm, chưa phải chuỗi báo cáo
quý (và VCB nhận thêm một ô chặn là KQKD 2008-Q4, quý lũy kế không bao giờ tách ra được). Đo cả
ba cách rồi mới chọn: **lọc liền mạch trước, rồi lấy filing quý sớm nhất CÒN LẠI trong chuỗi.**

⚠️ **Và lý do cũ viện ra cho BID cũng SAI.** Ghi chú cũ nói KQKD Q3-2011 "là lũy kế, BID không nộp
Q1/Q2-2011 nên không có gì để trừ → không thể tạo ra". `_decumulate` chỉ đụng vào period có
`half_year=True` (`cafef_financials.py`: `if q == 1 or not half_year.get(period): continue`); filing
Q3-2011 của BID mang `half_year=False`, nên nó **bị một gate từ chối** — parse hỏng thật, đúng như
`CLAUDE.md` §6-2 đã đính chính.

⚠️ **Bằng chứng "quý này CÓ filing" bắt buộc phải đến từ ngoài statements CSV.** Dòng `missing`
không mang `document` (§6-2-terdecies đã xoá provenance khỏi dòng missing), nên chỉ đọc CSV thì
"doanh nghiệp không nộp" và "có filing mà parse hỏng" là **một chữ giống hệt nhau**. Notebook gọi
thẳng `FinancialsBuilder.documents()` — *chính* bộ đọc mà `build()` dùng — nên luật chọn filing (hợp
nhất trước, mức soát xét sau, báo cáo năm đã kiểm toán thay cho Q4) không bị chép lại lần thứ hai.

⚠️ **"Còn thiếu" gồm hai loại và cả hai đều tính.** `missing` = có dòng, đã thử, bị từ chối.
`absent` = **không có dòng nào cả**, quý có filing mà lưới CSV chưa với tới. Loại thứ hai vô hình
với bản cũ, và nó không hiếm: VIC có **45 quý** như vậy (lần chạy authoritative bị dừng giữa chừng,
§6-2-duotricies) và BID có **Q2-2026** (CafeF công bố sau khi parse xong — *"một ticker đã parse
không đứng yên"*, và trước đây không có gì trong pipeline nhìn thấy điều đó). Cell 8 in riêng từng
loại.

⚠️ **`complete = False` nghĩa là *chưa chứng minh được liền mạch*, không phải *parser hỏng*.** Một
quý có filing mà thiếu vẫn có thể là filing không chứa statement đó, hoặc một dòng KQKD lũy kế mà
`pdf_ocr_merge` từ chối ghi — đều là "chưa xong", không phải "OCR sai".

⚠️ **Chỉ mục PDF KHÔNG nằm trong git** (`raw_data/` bị ignore trừ `raw_data/cafef/financials/`), nên
một bản checkout mới sẽ không có. Thiếu chỉ mục thì không chứng minh được quý nào có filing → ticker
đó `complete = False` kèm cảnh báo ở cell 6, chứ không đoán (`CLAUDE.md` §5 rule 2).

⚠️ Notebook này **chỉ đọc**, không parse, không OCR, không ghi gì vào `raw_data/`.


## 1 · Định vị thư mục

⚠️ Repo root được dò ngược từ thư mục hiện hành, nên notebook chạy được cả khi kernel mở ở
`src/kaggle_gpu/` lẫn ở repo root. Hai thư mục dữ liệu (`statements/`, `pdfs/index/`) đều lấy từ
**một mỏ neo duy nhất** — `pdf_ocr_job.use_data_root()`, đường chính thức để trỏ lại
`cafef_financials` — và được **in ra**.

⚠️ `CWD-1`: `fin.STATEMENTS_DIR` và `fin.PDFS_DIR` mặc định là đường dẫn **tương đối**, nên gọi từ
`src/kaggle_gpu/` sẽ đọc một thư mục rỗng — trông y hệt một ticker chưa parse bao giờ.


In [1]:
import sys
from pathlib import Path

import pandas as pd

REPORTS = ["balance_sheet", "income_statement", "cash_flow"]
REL_STATEMENTS = Path("raw_data/cafef/financials/statements")


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / REL_STATEMENTS).is_dir():
            return candidate
    raise FileNotFoundError(f"khong tim thay {REL_STATEMENTS} tu {here} tro len")


REPO_ROOT = _repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from web_scraper import cafef_financials as fin  # noqa: E402
from web_scraper import pdf_ocr_job as job  # noqa: E402

# CWD-1: `fin.STATEMENTS_DIR` / `fin.PDFS_DIR` mac dinh la duong dan TUONG DOI, nen goi tu
# `src/kaggle_gpu/` se doc mot thu muc rong — trong y het mot ticker chua parse bao gio.
# `use_data_root` la duong CHINH THUC de tro lai (no set ca nam hang so cua module cung mot luc),
# nen o day khong co ban sao thu hai cua bat ky duong dan nao.
DATA_ROOT = job.use_data_root(REPO_ROOT / "raw_data" / "cafef")
STATEMENTS_DIR = Path(fin.STATEMENTS_DIR)
PDF_INDEX_DIR = Path(fin.PDFS_DIR) / "index"

print("cwd            :", Path.cwd())
print("repo root      :", REPO_ROOT)
print("data root      :", DATA_ROOT)
print("statements dir :", STATEMENTS_DIR)
print("pdf index dir  :", PDF_INDEX_DIR, "" if PDF_INDEX_DIR.is_dir() else "<- KHONG CO")


cwd            : D:\GIT\master-thesis\src\kaggle_gpu
repo root      : D:\GIT\master-thesis
data root      : D:\GIT\master-thesis\raw_data\cafef
statements dir : D:\GIT\master-thesis\raw_data\cafef\financials\statements
pdf index dir  : D:\GIT\master-thesis\raw_data\cafef\pdfs\index 


## 2 · Đọc mọi CSV thành một bảng dài

Chỉ lấy các cột meta cần thiết. `encoding='utf-8-sig'` là bắt buộc — cột đầu tiên của các file này
mang BOM, nếu không sẽ thành `﻿symbol`.

⚠️ Cột `document` vẫn được đọc (để tra cứu), nhưng **không còn được dùng để suy ra quý nào có báo
cáo quý** — đó chính là chỗ bản cũ sai: dòng `missing` không mang `document`, nên tên file chỉ nói
được về những quý đã parse **thành công**. Bằng chứng filing đến từ cell 6.


In [2]:
META = ["symbol", "exchange", "template", "period", "year", "quarter", "source", "document"]

frames = []
for path in sorted(STATEMENTS_DIR.glob("*/*/*.csv")):
    report = path.parent.name
    if report not in REPORTS:
        print(f"WARNING: bo qua {path} — thu muc bao cao la {report!r}")
        continue
    part = pd.read_csv(path, usecols=META, encoding="utf-8-sig")
    part["report"] = report
    part["file"] = path.relative_to(REPO_ROOT).as_posix()
    frames.append(part)

if not frames:
    raise FileNotFoundError(f"khong co file .csv nao trong {STATEMENTS_DIR}")

records = pd.concat(frames, ignore_index=True)
records["source"] = records["source"].fillna("missing")


def quarter_rank(period: pd.Series) -> pd.Series:
    """`"Q3-2014"` -> 8059 — so nguyen tang dan, de lay min/max ma khong sap xep chuoi."""
    return period.str.slice(3).astype(int) * 4 + period.str.slice(1, 2).astype(int)


def quarter_id(period: pd.Series) -> pd.Series:
    """`"Q3-2014"` -> `"2014-Q3"` — sap xep duoc, va la dang `--quarters` cua `pdf_ocr_job`."""
    return period.str.slice(3) + "-" + period.str.slice(0, 2)


# `period` la thu duy nhat CA HAI nguon (statements CSV va chi muc PDF) deu mang, nen moi phep so
# sanh quy o duoi deu di qua no — mot luat, hai bang. Cot `year`/`quarter` cua CSV phai dong y:
# neu khong thi mot trong hai da bi doc nham cot, va im lang o day se thanh mot phep join sai o
# cell 8, cho ma ket qua sai van trong hop ly.
disagree = records["period"] != ("Q" + records["quarter"].astype(int).astype(str)
                                 + "-" + records["year"].astype(int).astype(str))
if disagree.any():
    raise ValueError(f"{int(disagree.sum())} dong co `period` khong khop `year`/`quarter`")

# mot symbol niem yet tren hai san se pha khoa ticker — doi khoa thay vi im lang gop nham
pairs = records[["exchange", "symbol"]].drop_duplicates()
_PREFIX_EXCHANGE = bool(pairs["symbol"].duplicated().any())
TICKER_BY_PAIR = {(e, s): (f"{e}_{s}" if _PREFIX_EXCHANGE else s)
                  for e, s in pairs.itertuples(index=False)}


def ticker_key(exchange: pd.Series, symbol: pd.Series) -> pd.Series:
    """Khoa ticker — MOT luat, dung cho ca `records` lan `filings`."""
    return pd.Series([TICKER_BY_PAIR[k] for k in zip(exchange, symbol)], index=symbol.index)


records["ticker"] = ticker_key(records["exchange"], records["symbol"])
records["rank"] = quarter_rank(records["period"])
records["quarter_id"] = quarter_id(records["period"])

# `(ticker, report, period)` la khoa ma luoi o cell 8 join vao. Mot dong trung se nhan ban luoi
# do va lam moi phep dem o duoi sai theo mot huong khong ai nhin thay.
duplicated = int(records.duplicated(["ticker", "report", "period"]).sum())
if duplicated:
    raise ValueError(f"{duplicated} dong trung khoa (ticker, report, period)")

unexpected = sorted(set(records["source"]) - {"pdf", "missing"})
if unexpected:
    print(f"WARNING: gia tri source ngoai du kien: {unexpected} — "
          f"bang duoi coi chung KHONG phai 'co bao cao'")

print(f"{len(frames)} file / {records['ticker'].nunique()} co phieu / {len(records)} dong")
print(records["source"].value_counts().to_dict())
records


21 file / 7 co phieu / 1182 dong
{'pdf': 1003, 'missing': 179}


,symbol,exchange,template,period,year,quarter,source,document,report,file,ticker,rank,quarter_id
0,ACB,HOSE,bank,Q1-2008,2008,1,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8033,2008-Q1
1,ACB,HOSE,bank,Q2-2008,2008,2,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8034,2008-Q2
2,ACB,HOSE,bank,Q3-2008,2008,3,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8035,2008-Q3
3,ACB,HOSE,bank,Q4-2008,2008,4,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8036,2008-Q4
4,ACB,HOSE,bank,Q1-2009,2009,1,missing,NaN,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8037,2009-Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1177,VIC,HOSE,corp,Q4-2013,2013,4,missing,NaN,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8056,2013-Q4
1178,VIC,HOSE,corp,Q1-2014,2014,1,pdf,Q1-2014_bao_cao_tai_chinh_hop_nhat_quy_1_nam_2...,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8057,2014-Q1
1179,VIC,HOSE,corp,Q2-2014,2014,2,pdf,Q2-2014_bao_cao_tai_chinh_hop_nhat_quy_2_nam_2...,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8058,2014-Q2
1180,VIC,HOSE,corp,Q3-2014,2014,3,pdf,Q3-2014_bao_cao_tai_chinh_hop_nhat_quy_3_nam_2...,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8059,2014-Q3


## 3 · Chỉ mục PDF — quý nào **thực sự** có filing

Đây là nửa mà bản trước không có, và là nửa quyết định: một quý `missing` chỉ đáng gọi là *thiếu*
khi doanh nghiệp **có nộp** báo cáo cho quý đó. Statements CSV không trả lời được câu này — dòng
`missing` không mang `document` — nên bằng chứng phải lấy từ chỉ mục PDF.

⚠️ **Gọi thẳng `FinancialsBuilder.documents()`, không chép lại luật chọn filing.** Luật đó không hề
tầm thường (hợp nhất trước rồi mới tới mức soát xét; báo cáo năm đã kiểm toán đứng thay cho Q4
nhưng **không bao giờ đổi entity**; sàn `FINANCIALS_PERIOD_MIN = Q1-2008`), và một bản sao thứ hai
sẽ lệch khỏi bản gốc ngay lần đầu bản gốc đổi.

⚠️ **`allow_parent=True` — rộng nhất trong những gì có trên đĩa.** Một quý chỉ có báo cáo riêng lẻ
vẫn là một filing **chưa parse**; đếm nó vào là bảo thủ đúng hướng, không phải buộc tội oan. Một
ticker từng chạy ở chế độ *chỉ hợp nhất* sẽ hiện ra là còn việc phải làm — và đúng là còn thật.

⚠️ **SỬA 2026-08-30 — `quarterly_filing` ĐỌC TỪ CHỈ MỤC, KHÔNG TỪ FILING ĐƯỢC CHỌN.** Bản trước
lấy `annual == "False"` của tài liệu `documents()` trả về; nhưng `documents()` trả về **đúng một
filing mỗi quý** và ưu tiên bản báo cáo **năm** đã kiểm toán ở Q4, nên cờ đó trả lời *"filing
ĐƯỢC CHỌN có phải báo cáo quý không"* — không phải *"doanh nghiệp có nộp báo cáo quý không"*.
**ACB 2009-Q4 là phản ví dụ nằm ngay trong bộ dữ liệu này**: chỉ mục mang cả *"Báo cáo tài chính
quý 4 năm 2009"* (`quarter=4`) lẫn bản năm đã kiểm toán (`quarter=5`), bản năm thắng, và cột cũ
trả lời `False` cho một quý mà ACB **có** nộp báo cáo quý. Cột bây giờ hỏi thẳng cột `quarter`
của chỉ mục — **98 quý đổi `False` → `True` trên 7 cổ phiếu, 0 quý đổi ngược lại**, và chiều
ngược ("`documents()` gọi là báo cáo quý ⇒ chỉ mục phải có dòng `quarter` 1..4") được **assert**
ở cell 6, 0 vi phạm.

⚠️ **Không một `first_report` nào của 7 cổ phiếu hôm nay đổi chỗ** — mọi chuỗi hiện tại đều bắt
đầu ở một quý không phải Q4. Đây là sửa một lỗi **tiềm ẩn**: cổ phiếu nào có chuỗi bắt đầu đúng
ở một Q4 vừa có báo cáo quý vừa có bản năm sẽ bị đẩy mốc trễ một quý và **ba ô lặng lẽ rời khỏi
mẫu số** — đúng hình dạng notebook này đã phải sửa hai lần (mốc lấy từ chính kết quả parse; một
filing lẻ loi mở đầu chuỗi). **Mốc phải đọc từ SỰ KIỆN của chỉ mục, không từ một lựa chọn ở
bước sau.** Một năm chỉ có báo cáo năm thì chuỗi báo cáo quý vẫn chưa bắt đầu — đó là ý định cũ,
và bây giờ mã mới làm đúng nó.

⚠️ **Và cell 6 đo thêm `parent_only`** — quý mà bản duy nhất tồn tại là báo cáo **riêng lẻ**, tức
quý `pdf_ocr_job.plan()` ở mặc định `allow_parent=False` **không nhìn thấy** (nó báo *"files no
document for quarter(s) [...]"* trong khi file PDF nằm ngay trên đĩa). Đo 2026-08-30: ACB 4 quý
(2008-Q1, 2009-Q2, 2009-Q3, 2009-Q4 — **đúng cả bốn quý còn thiếu của ACB**), TCB 2, VIC 1, còn
BID/CTG/VCB/BSR 0. Bảng này đếm chúng vào mẫu số vì chúng là filing chưa đọc; danh sách in dưới
bảng ở cell 8 nói rõ phải bật nút nào mới mở được.

⚠️ **Nhưng phép LIỀN MẠCH ở cell 8 đo trên `filings`, tức MỌI filing, không riêng cột này.** Q4 là
`annual` gần như mọi năm, nên đòi liền mạch trên riêng filing quý sẽ đứt ở mỗi Q4 và chuỗi dài
nhất còn ba quý. Hai cột trả lời hai câu khác nhau: `rank` của `filings` trả lời *"quý này có
filing để đọc không"* (mẫu số của lưới), còn `quarterly_filing` trả lời *"chuỗi báo cáo quý đã bắt
đầu chưa"* (vị trí của mốc).


In [3]:
import csv  # noqa: E402

builder = fin.FinancialsBuilder()


def quarterly_filed(exchange: str, symbol: str) -> set:
    """Nhung quy ma ticker nay CO NOP mot bao cao QUY — doc thang cot `quarter` cua chi muc.

    ⚠️ **KHONG DOC DUOC DIEU NAY TU `documents()`, VA DO LA LOI DA SUA 2026-08-30.**
    `documents()` tra ve DUNG MOT filing moi quy va uu tien ban bao cao NAM da kiem toan o
    Q4, nen co `annual` cua no tra loi *"filing DUOC CHON co phai bao cao quy khong"* — khong
    phai *"ticker co nop bao cao quy khong"*. Hai cau khac nhau, va **ACB 2009-Q4 la vi du
    ngay trong bo du lieu nay**: chi muc mang ca "Bao cao tai chinh quy 4 nam 2009" (quarter=4)
    lan ban nam da kiem toan (quarter=5), `documents()` chon ban nam, va cot cu tra loi
    `False` cho mot quy MA ACB CO nop bao cao quy. Do 2026-08-30 tren 7 ticker: **98 quy doi
    tu False sang True**, khong quy nao doi nguoc lai.

    ⚠️ **Khong mot `first_report` nao trong 7 ticker hom nay doi cho** — moi chuoi hien tai
    deu bat dau o mot quy khong phai Q4. Day la sua mot loi TIEM AN: ticker nao co chuoi bat
    dau dung o mot Q4 vua co bao cao quy vua co ban nam se bi day moc tre mot quy va **ba o
    lang le roi khoi mau so**, dung hinh dang ma notebook nay da phai sua hai lan (moc lay tu
    ket qua parse; mot filing le loi mo dau chuoi). MOC PHAI DOC TU SU KIEN CUA CHI MUC.

    ⚠️ **Khong loc `consolidated`:** mau so o cell nay la `allow_parent=True`, nen phep thu
    "co bao cao quy khong" phai rong dung bang the — loc hep hon se lam ACB 2009-Q4 (bao cao
    RIENG LE) rot lai. Day la mot CAU HOI KHAC chu khong phai ban sao cua luat chon filing
    (hop nhat truoc, muc soat xet sau); luat do van chi nam trong `documents()`.
    """
    with open(PDF_INDEX_DIR / f"{exchange}_{symbol}.csv", encoding="utf-8-sig") as f:
        return {f"Q{int(r['quarter'])}-{r['year']}" for r in csv.DictReader(f)
                if str(r["quarter"]).isdigit() and int(r["quarter"]) in (1, 2, 3, 4)}


rows, no_index = [], []
for listed_on, symbol in records[["exchange", "symbol"]].drop_duplicates().itertuples(index=False):
    try:
        docs = builder.documents(listed_on, symbol, allow_parent=True)
    except FileNotFoundError:
        no_index.append(f"{listed_on}_{symbol}")
        continue
    quarterly = quarterly_filed(listed_on, symbol)
    # Chieu nguoc lai phai luon dung: mot filing ma `documents()` goi la bao cao quy thi trong
    # chi muc BAT BUOC co mot dong quarter 1..4. Neu khong, hai ben dang doc chi muc theo hai
    # luat khac nhau va cot `quarterly_filing` se HEP hon su that ma khong ai thay.
    # (Do 2026-08-30: 0 vi pham tren ca 7 ticker.)
    stale = [d["period"] for d in docs
             if d["annual"] == "False" and d["period"] not in quarterly]
    if stale:
        raise ValueError(f"{listed_on}_{symbol}: {len(stale)} filing quy cua `documents()` "
                         f"khong co dong quarter 1..4 nao trong chi muc: {stale[:5]}")
    # ⚠️ CUNG MOT CAU HOI, O DO RONG MAC DINH CUA CONG CU PARSE. Chenh lech giua hai ben la
    # nhung quy CHI ton tai duoi dang bao cao RIENG LE, va do la cay cau giua hai notebook:
    # `pdf_ocr_job.plan()` o `allow_parent=False` bao "files no document for quarter(s) [...]"
    # trong khi file PDF nam ngay tren dia. ACB 2009-Q2/Q3/Q4 la dung ba quy do.
    consolidated_only = {d["period"] for d in
                         builder.documents(listed_on, symbol, allow_parent=False)}
    rows += [(symbol, listed_on, d["period"], d["period"] in quarterly,
              d["period"] not in consolidated_only) for d in docs]

if no_index:
    print("WARNING: khong co chi muc PDF cho:", ", ".join(no_index))
    print("         -> khong chung minh duoc quy nao co filing, nen cac ticker nay luon "
          "`complete=False` (§5 rule 2: thieu phep do thi la thieu, khong suy dien)")

filings = pd.DataFrame(rows, columns=["symbol", "exchange", "period", "quarterly_filing",
                                      "parent_only"])
if filings.empty:
    raise FileNotFoundError(f"khong doc duoc chi muc PDF nao trong {PDF_INDEX_DIR}")
filings["ticker"] = ticker_key(filings["exchange"], filings["symbol"])
filings["rank"] = quarter_rank(filings["period"])
filings["quarter_id"] = quarter_id(filings["period"])

# `documents()` tra ve dung mot filing moi quy (no gom theo `period`), va luoi o cell 8 dua vao
# dieu do — neu no doi, phep dem duoi kia phong len ma khong bao gi.
duplicated = int(filings.duplicated(["ticker", "period"]).sum())
if duplicated:
    raise ValueError(f"{duplicated} filing trung khoa (ticker, period)")

# Kiem tra nguoc, va no re: moi dong `pdf` phai roi vao mot quy CO filing trong chi muc. Neu khong
# thi hai nguon dang noi hai chuyen khac nhau va bang o cell 8 khong doc duoc.
filed = set(map(tuple, filings[["ticker", "period"]].itertuples(index=False)))
orphan = sorted({(t, p) for t, p, s in zip(records["ticker"], records["period"], records["source"])
                 if s == "pdf" and (t, p) not in filed})
if orphan:
    print(f"WARNING: {len(orphan)} quy co dong `pdf` ma chi muc khong co filing nao:", orphan[:10])

print(f"{filings['ticker'].nunique()} co phieu / {len(filings)} filing / "
      f"{int(filings['quarterly_filing'].sum())} filing quy / "
      f"{int(filings['parent_only'].sum())} quy CHI co bao cao rieng le "
      f"(can ALLOW_PARENT=True moi mo duoc)")
filings


7 co phieu / 419 filing / 405 filing quy / 7 quy CHI co bao cao rieng le (can ALLOW_PARENT=True moi mo duoc)


,symbol,exchange,period,quarterly_filing,parent_only,ticker,rank,quarter_id
0,ACB,HOSE,Q1-2008,True,True,ACB,8033,2008-Q1
1,ACB,HOSE,Q2-2009,True,True,ACB,8038,2009-Q2
2,ACB,HOSE,Q3-2009,True,True,ACB,8039,2009-Q3
3,ACB,HOSE,Q4-2009,True,True,ACB,8040,2009-Q4
4,ACB,HOSE,Q1-2010,True,False,ACB,8041,2010-Q1
...,...,...,...,...,...,...,...,...
414,VIC,HOSE,Q1-2025,True,False,VIC,8101,2025-Q1
415,VIC,HOSE,Q2-2025,True,False,VIC,8102,2025-Q2
416,VIC,HOSE,Q3-2025,True,False,VIC,8103,2025-Q3
417,VIC,HOSE,Q4-2025,True,False,VIC,8104,2025-Q4


## 4 · Bảng tóm tắt — 6 cột

`exchange` là sàn của cổ phiếu. `first_report` là **quý mở đầu chuỗi filing liền mạch** trong chỉ
mục PDF — đi ngược từ filing mới nhất của cổ phiếu đó cho tới quý đầu tiên không có filing, rồi
lấy filing **quý** sớm nhất còn nằm trong chuỗi; ba cột cuối là **quý muộn nhất còn thiếu** của
từng statement.

`complete` đứng ngay sau `exchange`, kiểu `bool` (không phải chuỗi, để lọc thẳng bằng
`summary[summary["complete"]]`): **kể từ `first_report` trở đi, mọi quý CÓ FILING đều đã có dòng
`pdf` ở cả ba statement.**

Lưới kỳ vọng là `mọi quý có filing × ba statement`. Một ô chưa có dòng `pdf` là một ô **còn
thiếu**, và nó có đúng hai dạng:

| dạng | nghĩa | ví dụ |
|---|---|---|
| `missing` | có dòng trong CSV, đã thử, bị một gate từ chối hoặc `pdf_ocr_merge` không ghi | BID KQKD **2011-Q3** |
| `absent` | **không có dòng nào** — lưới CSV chưa với tới quý đó | VIC 45 quý sau 2014-Q4; BID **2026-Q2** |

⚠️ **Mốc bắt đầu là `first_report` của cả ticker và đến từ chỉ mục PDF, không từ kết quả parse.**
Đây là điểm sửa thứ nhất: lấy mốc từ quý đầu tiên *parse được* thì mọi lần hỏng ở đầu chuỗi tự đẩy
mốc qua chính nó rồi biến mất — BID `complete=True` trong khi KQKD 2011-Q3 hỏng, đúng cái filing
mà hai statement kia đọc được.

⚠️ **Và mốc phải LIỀN MẠCH — điểm sửa thứ hai.** Chuỗi đi ngược từ filing mới nhất và **đứt ngay ở
quý đầu tiên không có filing**; filing nằm trước chỗ đứt không mở đầu chuỗi nào cả. ACB có đúng
một filing 2008-Q1 rồi trống bốn quý: mốc cũ là 2008-Q1, nên ba ô của một quý đơn độc bị kéo vào
mẫu số suốt từ đó. Mốc mới là **2009-Q2**.

⚠️ **Và mốc đọc từ chỉ mục chứ không từ filing ĐƯỢC CHỌN — điểm sửa thứ ba, 2026-08-30.**
`documents()` trả về một filing mỗi quý và ưu tiên bản năm đã kiểm toán ở Q4, nên cờ `annual` của
nó không trả lời được câu *"doanh nghiệp có nộp báo cáo quý cho quý này không"*. ACB 2009-Q4 nộp
cả hai bản và cột cũ vẫn trả lời `False`. Không mốc nào của 7 cổ phiếu hôm nay đổi chỗ (98 quý
đổi cờ, 0 mốc đổi) — đây là sửa lỗi **tiềm ẩn**, chi tiết ở cell 3.

Ba cột cuối là **quý muộn nhất còn thiếu**, tính trên toàn bộ lịch sử, nên một ô nằm **trước**
`first_report` không chặn `complete`; muốn biết quý nào đang chặn thì đọc phần in ra dưới bảng —
nó liệt kê theo từng statement, dán thẳng vào `--quarters` được, và **nói rõ quý nào chỉ có báo
cáo riêng lẻ, tức cần `ALLOW_PARENT=True`**. ⚠️ Đoạn in đó bị comment từ lúc được viết cho tới
2026-08-30, trong khi đoạn văn này vẫn hứa nó — một tài liệu hứa một đầu ra không tồn tại.

⚠️ **Mẫu số là filing, không phải toàn bộ lịch.** Quý không ai nộp thì không nằm trong mẫu số —
đúng phép re-scope `CLAUDE.md` §6-2-quindecies đã làm cho BID (bỏ lịch nộp ra khỏi mẫu số thì
80.0 % thành 94.2 %). Khác với bản cũ, phép loại trừ này giờ dựa trên **bằng chứng** (chỉ mục PDF)
chứ không dựa trên việc parse có thành công hay không.

⚠️ **`complete = False` là *chưa chứng minh được liền mạch*, không phải *parser hỏng*.** Một filing
có thể không chứa statement đó; một dòng KQKD lũy kế mà không có Q1..Q(q-1) để trừ sẽ bị
`_decumulate` bỏ và `pdf_ocr_merge` từ chối ghi (CTG có 32 quý Q2/Q4 đúng dạng này). Cả hai đều là
việc còn phải làm, không phải số sai.

⚠️ **Ticker nào chưa có filing quý nào trong chỉ mục thì là `False`:** không biết chuỗi bắt đầu từ
đâu thì không thể nói nó liền mạch.


In [4]:
import textwrap

exchange_of = records.groupby("ticker")["exchange"].first().rename("exchange")
index = pd.Index(sorted(records["ticker"].unique()), name="ticker")

# MOC BAT DAU LAY TU CHI MUC PDF, KHONG TU KET QUA PARSE. Lay tu ket qua parse thi moi lan hong
# nam o DAU chuoi se tu day moc qua chinh no va khong bao gio dem duoc — do la loi da sua: BID
# `complete=True` trong khi KQKD 2011-Q3 hong, dung cai filing ma hai statement kia doc duoc.


def chain_start(rank: pd.Series) -> int:
    """Quy MO DAU chuoi filing lien mach, di nguoc tu filing MOI NHAT cua chinh ticker do.

    Cham vao mot quy khong co filing nao la DUT: nhung gi nam truoc cho dut khong thuoc chuoi
    nua, du van con filing le te o do. Neo o filing moi nhat CUA TICKER chu khong o quy lich
    hien tai — mot ma da huy niem yet van co chuoi cua rieng no, va lay quy hien tai lam neo se
    lam moi ma dut ngay o buoc dau.

    `(ticker, period)` da duoc khang dinh khong trung o cell 6, nen ranks tang deu tung 1 va lui
    mot buoc mot la du.
    """
    ranks = sorted(rank)
    start = ranks[-1]
    for r in reversed(ranks[:-1]):
        if r != start - 1:
            break
        start = r
    return start


# LIEN MACH DO TREN MOI FILING, khong rieng filing quy: `documents()` gop bao cao NAM da kiem
# toan vao Q4, nen doi lien mach tren rieng cot `quarterly_filing` se dut o MOI Q4 va chuoi dai
# nhat chi con ba quy.
chain = filings.groupby("ticker")["rank"].apply(chain_start)

# ... roi trong chuoi do, moc la filing QUY som nhat: mot nam chi co bao cao nam thi chuoi bao
# cao quy chua bat dau. Hai luat, moi luat mot ly do, va LOC LIEN MACH CHAY TRUOC — lay thang
# quy dau chuoi lam moc thi VCB/CTG lui ve 2008-Q4 (nam do chi co bao cao nam) va VCB nhan them
# mot o chan la KQKD 2008-Q4, quy luy ke khong bao gio tach ra duoc.
quarterly = (filings[filings["quarterly_filing"]
                     & filings["rank"].ge(filings["ticker"].map(chain))]
             .sort_values("rank"))
first_rank = quarterly.groupby("ticker")["rank"].min().reindex(index)
first_report = (quarterly.groupby("ticker")["quarter_id"].first()
                .reindex(index).rename("first_report"))

# LUOI KY VONG: moi quy CO FILING x ba statement. Mot o chua co dong `pdf` la mot o CON THIEU, va
# `missing` (co dong, da thu, bi tu choi) lan `absent` (khong co dong nao) deu tinh — dem mot loai
# ma bo loai kia chinh la cach mot lan chay bi dung giua chung trong nhu da xong (VIC, 45 quy).
expected = filings[["ticker", "period", "rank", "quarter_id", "parent_only"]].merge(
    pd.DataFrame({"report": REPORTS}), how="cross")
expected = expected.merge(records[["ticker", "period", "report", "source"]],
                          how="left", on=["ticker", "period", "report"])
expected["state"] = expected["source"].fillna("absent")
outstanding = expected[expected["state"] != "pdf"]

# So sanh voi NaN tra ve False, nen mot ticker CHUA co filing quy nao se khong co dong nao bi chan
# va lot qua thanh `True` — phai chan rieng bang `notna()`.
blocking = outstanding[outstanding["rank"].ge(outstanding["ticker"].map(first_rank))]
complete = pd.Series(~index.isin(blocking["ticker"].unique()) & first_rank.notna().to_numpy(),
                     index=index, name="complete")

last_outstanding = (outstanding.sort_values("rank")
                    .groupby(["ticker", "report"])["quarter_id"].last()
                    .unstack("report").reindex(index=index, columns=REPORTS))

summary = pd.DataFrame(index=index).join(exchange_of).join(first_report).join(last_outstanding)
text_cols = ["first_report", *REPORTS]
summary[text_cols] = summary[text_cols].fillna("—")
# `complete` join SAU fillna, de no giu kieu bool thay vi bi doi thanh chuoi
summary = summary.join(complete)[["exchange", "complete", *text_cols]]

print(f"complete: {int(complete.sum())}/{len(index)} co phieu lien mach ke tu `first_report`")
blind = sorted(index[first_rank.isna()])
if blind:
    print("WARNING: chuoi lien mach khong chua filing quy nao (hoac thieu chi muc):",
          ", ".join(blind))
# ⚠️ IN RA, KHONG COMMENT LAI. Doan nay bi comment tu chinh luc no duoc viet, trong khi phan
# markdown ngay tren van hua "muon biet quy nao dang chan thi doc phan in ra duoi bang" — mot
# tai lieu hua mot dau ra khong ton tai, va nguoi doc chi con 6 cot de doan tiep.
if not blocking.empty:
    # In DU, khong cat bot: day dung la dang `--quarters` cua `pdf_ocr_job`, nen day la thu duy
    # nhat trong notebook nay dan thang vao mot lenh parse duoc.
    print("\nquy dang CHAN `complete` — co filing, tu `first_report` tro di, chua co dong `pdf`:")
    for (tkr, report), group in blocking.groupby(["ticker", "report"]):
        quarters = sorted(group["quarter_id"])
        print(f"  {tkr} {report} [{len(quarters)}] {group['state'].value_counts().to_dict()}")
        print(textwrap.fill(" ".join(quarters), width=96,
                            initial_indent="    ", subsequent_indent="    "))
        # ⚠️ VA NOI RO CAI NUT PHAI BAT. Mau so o bang nay la `allow_parent=True`, con
        # `pdf_ocr_job` mac dinh `False` — nen mot quy chi co bao cao RIENG LE vua bi bao
        # thieu o day vua bi `plan()` tra loi "files no document for quarter(s) [...]" o
        # notebook parse, ve dung mot quy ma file PDF dang nam tren dia. ACB 2009-Q2/Q3/Q4 la
        # ba quy do; doan nay la cho noi hai cau tra loi lai voi nhau.
        parent = sorted(group.loc[group["parent_only"], "quarter_id"])
        if parent:
            print(f"    ^ {len(parent)} quy CHI co bao cao rieng le — dat ALLOW_PARENT=True "
                  f"trong RUN__pdf_ocr_control*.ipynb, neu khong `plan()` bao la khong co "
                  f"filing: {' '.join(parent)}")

summary


complete: 0/7 co phieu lien mach ke tu `first_report`

quy dang CHAN `complete` — co filing, tu `first_report` tro di, chua co dong `pdf`:


  ACB balance_sheet [1] {'missing': 1}
    2009-Q3
    ^ 1 quy CHI co bao cao rieng le — dat ALLOW_PARENT=True trong RUN__pdf_ocr_control*.ipynb, neu khong `plan()` bao la khong co filing: 2009-Q3
  ACB cash_flow [2] {'missing': 2}
    2009-Q2 2009-Q3
    ^ 2 quy CHI co bao cao rieng le — dat ALLOW_PARENT=True trong RUN__pdf_ocr_control*.ipynb, neu khong `plan()` bao la khong co filing: 2009-Q2 2009-Q3
  ACB income_statement [2] {'missing': 2}
    2009-Q2 2009-Q4
    ^ 2 quy CHI co bao cao rieng le — dat ALLOW_PARENT=True trong RUN__pdf_ocr_control*.ipynb, neu khong `plan()` bao la khong co filing: 2009-Q2 2009-Q4
  BID balance_sheet [1] {'absent': 1}
    2026-Q2
  BID cash_flow [1] {'absent': 1}
    2026-Q2
  BID income_statement [1] {'absent': 1}
    2026-Q2
  BSR balance_sheet [2] {'missing': 2}
    2018-Q3 2019-Q4
  BSR cash_flow [1] {'missing': 1}
    2018-Q2
  BSR income_statement [7] {'missing': 7}
    2018-Q2 2018-Q3 2018-Q4 2019-Q2 2019-Q4 2020-Q2 2020-Q4
  CTG balance_sheet [

,exchange,complete,first_report,balance_sheet,income_statement,cash_flow
ticker,,,,,,
ACB,HOSE,False,2009-Q2,2009-Q3,2009-Q4,2009-Q3
BID,HOSE,False,2011-Q3,2026-Q2,2026-Q2,2026-Q2
BSR,HOSE,False,2018-Q2,2019-Q4,2020-Q4,2018-Q2
CTG,HOSE,False,2009-Q1,2019-Q1,2025-Q4,2024-Q1
TCB,HOSE,False,2012-Q2,2013-Q1,2012-Q3,2021-Q1
VCB,HOSE,False,2009-Q1,2009-Q2,2008-Q4,2009-Q2
VIC,HOSE,False,2008-Q2,2026-Q1,2026-Q1,2026-Q1


## 5 · Phụ — đếm quý đã parse / còn thiếu

Không nằm trong 6 cột được hỏi, nhưng là mẫu số để đọc bảng trên: một `first_report` sớm mà
`coverage` thấp nghĩa là chuỗi dài nhưng rỗng, chứ không phải lịch sử dài.

⚠️ **`coverage` chia cho `filed`, không chia cho số dòng CSV.** `rows` là khoảng thời gian file phủ,
còn `filed` là số quý **thực sự có filing** — hai số này lệch nhau ở cả hai đầu: quý không ai nộp
làm `rows` phồng lên, còn quý có filing mà lưới CSV chưa với tới (`outstanding` dạng `absent`) thì
không có dòng nào để đếm. Chia cho `rows` là trộn lịch nộp báo cáo vào phép đo parser.


In [5]:
tally = (
    records.groupby(["ticker", "report"])["source"]
    .value_counts()
    .unstack("source")
    .reindex(columns=["pdf", "missing"])
    .fillna(0)
    .astype(int)
)
tally["rows"] = tally["pdf"] + tally["missing"]
tally["filed"] = tally.index.get_level_values("ticker").map(filings.groupby("ticker").size())
tally["outstanding"] = (outstanding.groupby(["ticker", "report"]).size()
                        .reindex(tally.index).fillna(0).astype(int))
tally["coverage"] = (tally["pdf"] / tally["filed"]).round(3)
tally


source                   pdf  missing  rows  filed  outstanding  coverage
ticker report                                                            
ACB    balance_sheet      67        6    73     69            2     0.971
       cash_flow          66        7    73     69            3     0.957
       income_statement   66        7    73     69            3     0.957
BID    balance_sheet      62        8    70     63            1     0.984
       cash_flow          61        9    70     63            2     0.968
       income_statement   59       11    70     63            4     0.937
BSR    balance_sheet      10        7    17     14            4     0.714
       cash_flow          13        4    17     14            1     0.929
       income_statement    6       11    17     14            8     0.429
CTG    balance_sheet      61        9    70     70            9     0.871
       cash_flow          61        9    70     70            9     0.871
       income_statement   35       35    70     70           35     0.500
TCB    balance_sheet      57       10    67     61            4     0.934
       cash_flow          52       15    67     61            9     0.852
       income_statement   59        8    67     61            2     0.967
VCB    balance_sheet      68        2    70     70            2     0.971
       cash_flow          69        1    70     70            1     0.986
       income_statement   69        1    70     70            1     0.986
VIC    balance_sheet      20        7    27     72           52     0.278
       cash_flow          20        7    27     72           52     0.278
       income_statement   22        5    27     72           50     0.306